### Google Drive Specific Cells

In [ ]:
"""
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/CS757/retnet_hf" /content/ -r
#from retnet_hf import RetNetConfig, RetNetForCausalLM
#!mkdir output
!ls .
!pip install torchscale
!pip install fairscale
!cp "/content/drive/MyDrive/CS757/retnet_hf/multiscale_retention.py" "/usr/local/lib/python3.12/dist-packages/torchscale/component/multiscale_retention.py"
"""

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive  retnet_hf  sample_data


### Environment Variable

In [ ]:
#Get CPU Count to speed up the tokenize part
import multiprocessing
num_cpu_proc = max (6, multiprocessing.cpu_count())

#Configure Batch_Size and learning rate based on GPU Vram.
from types import SimpleNamespace
args = SimpleNamespace() #Initialize
import torch
available_gpu_memory = torch.cuda.mem_get_info()[0] / (1024**3) #Get available gpu memory in Byte, convert to GB.
print ("available_gpu_memory: ", available_gpu_memory)
args.__dict__.update({
    "per_device_train_batch_size":round(available_gpu_memory),
    "per_device_eval_batch_size":round(available_gpu_memory),
    "gradient_accumulation_steps":2,
    "bf16": False,
    "fp16": True,
    #epoch/lr/weight decay
    "num_train_epochs": 1 * round(available_gpu_memory**0.4), #Feng Shui told me
    "learning_rate": 1e-4 * round(available_gpu_memory**0.5), #Feng Shui told me
    "weight_decay": 0.01,
})

available_gpu_memory:  14.46075439453125


In [ ]:
args.__dict__.update({
    #Logging Location
    "dataset_name": "FiscalNote/billsum",
    "output_dir": "/content/drive/MyDrive/CS757/model_checkpoint/retnet-billsum",

    #Logging/Saving/Evaluating
    "logging_steps": 20,
    "save_steps": 2500,
    "eval_steps": 200,
    "dataset_config_name": None,
    "dataset_text_field": None,
    "streaming": False,

    #Basic training config
    "max_steps": -1,
    "warmup_ratio": 0.03,
    "tokenizer_name": "gpt2",
    "dropout": 0.1,
    "activation_dropout": 0.0,

    #For trainer use only
    "block_size": 1024, #Chunking
    "max_train_samples": None,
    "max_eval_samples": 1024,
    "validation_split_percentage": 5,
    "seed": 42,
    "push_to_hub": False,
    "hub_model_id": None,

    #Model config
    "hidden_size": 1024,
    "intermediate_size": 2048,
    "num_hidden_layers": 8,
    "num_retention_heads": 8,
    "value_dim": None,
    "drop_path_rate": 0.0,
    "recurrent_chunk_size": 128,
    "chunkwise_recurrent": False,
})

### Import and basic functions

In [ ]:
#!/usr/bin/env python
from __future__ import annotations

import argparse
import logging
import math
import os
from functools import partial

from datasets import load_dataset
import transformers
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

from retnet_hf import RetNetConfig, RetNetForCausalLM
logger = logging.getLogger(__name__)

def infer_text_field(dataset_name: str, dataset_text_field: str | None) -> str | None:
    if dataset_text_field:
        return dataset_text_field
    lower = dataset_name.lower()
    if "billsum" in lower:
        return None  # handled by formatter below
    if "c4" in lower:
        return "text"
    return "text"

def load_raw_datasets(args: argparse.Namespace):
    lower = args.dataset_name.lower()
    if "billsum" in lower:
        ds = load_dataset(args.dataset_name, args.dataset_config_name)
        if "validation" in ds:
            train_ds, eval_ds = ds["train"], ds["validation"]
        else:
            if args.streaming:
                raise ValueError("Streaming Billsum validation split creation is not supported in this example.")
            split = ds["train"].train_test_split(
                test_size=args.validation_split_percentage / 100.0,
                seed=args.seed,
            )
            train_ds, eval_ds = split["train"], split["test"]
        return train_ds, eval_ds

    if "c4" in lower:
        config_name = args.dataset_config_name or "en"
        train_ds = load_dataset(args.dataset_name, config_name, split="train", streaming=args.streaming)
        eval_ds = load_dataset(args.dataset_name, config_name, split="validation", streaming=args.streaming)
        return train_ds, eval_ds

    ds = load_dataset(args.dataset_name, args.dataset_config_name)
    if "validation" in ds:
        return ds["train"], ds["validation"]
    if args.streaming:
        raise ValueError("Streaming is only wired here for datasets that already expose validation splits.")
    split = ds["train"].train_test_split(
        test_size=args.validation_split_percentage / 100.0,
        seed=args.seed,
    )
    return split["train"], split["test"]

## Append "text: " and "summary: " to the dataset to shove into causal_lm.
def format_for_causal_lm(example, dataset_name: str, text_field: str | None):
    lower = dataset_name.lower()
    if "billsum" in lower:
        #print ("billsum dataset detected")
        return {
            "text": f"Document:\n{example['text']}\n\nSummary:\n{example['summary']}"
        }
    if text_field is None:
        raise ValueError("Could not infer dataset text field. Pass --dataset_text_field.")
    return {"text": example[text_field]}

## Simply running the tokenizer
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"])

#Does not do fixed length block chunking anymore, just simply truncation and return fixed sample.
def group_texts(examples, block_size: int):
    result = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
    }

    #print (type(examples["input_ids"]))
    #print (len(examples["input_ids"]))
    #print (examples.keys)
    #for i in range(len(examples["input_ids"])):
    ids = examples["input_ids"]#[i]
    mask = examples.get("attention_mask", [None] * len(ids))

    # truncate
    #ids = ids[:block_size] #Get first n element
    ids = ids[-block_size:] #Get LAST n element
    if mask is not None:
        #mask = mask[:block_size]
        mask = mask[-block_size:] #Get LAST n element
    else:
        mask = [1] * len(ids)

    # pad if needed
    pad_len = block_size - len(ids)
    if pad_len > 0:
        ids = ids + [tokenizer.pad_token_id] * pad_len
        mask = mask + [0] * pad_len

    result["input_ids"].append(ids)
    result["attention_mask"].append(mask)
    result["labels"].append(ids.copy())

    return result

print ("DEBUGGING")
group_fn = partial(group_texts, block_size=args.block_size)
#temp = train_ds.map(group_fn, num_proc=num_cpu_proc)#, batched=True)
print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))

DEBUGGING
Document:
SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention 
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal 
year does not become law prior to the beginning of these fiscal years 
or a joint resolution making continuing appropriations is not in 
effect, there is appropriated, out of any moneys in the Treasury not 
otherwise appropriated, and out of applicable corporate or other 
revenues, receipts, and funds, such sums as may be necessary to 
continue any program, project, or activity for which funds were 
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and 
authority granted, for a program, project, or activity for any fiscal 
year pursuant to this Act shall be at 100 percent of the rate of 
operations that was provided for the program, project, or activity in 
fiscal year 1999 in the corresponding regular appropriation Act 

In [ ]:

## Turn dataset into a continous stream, divided by blocks.
"""def group_texts(examples, block_size: int):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = list(result["input_ids"])
    return result"""

'def group_texts(examples, block_size: int):\n    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}\n    total_length = len(concatenated_examples["input_ids"])\n    total_length = (total_length // block_size) * block_size\n    result = {\n        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]\n        for k, t in concatenated_examples.items()\n    }\n    result["labels"] = list(result["input_ids"])\n    return result'

In [ ]:
#args = parse_args()
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    level=logging.INFO,
)
set_seed(args.seed)

In [ ]:
text_field = infer_text_field(args.dataset_name, args.dataset_text_field)
train_raw, eval_raw = load_raw_datasets(args)

#GET SUBSET ONLY
train_raw = train_raw.select(range(min(50, len(train_raw))))
eval_raw = eval_raw.select(range(min(50, len(train_raw))))

tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
tokenizer.model_max_length = 2048

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

format_fn = partial(format_for_causal_lm, dataset_name=args.dataset_name, text_field=text_field)

remove_columns = getattr(train_raw, "column_names", None)
train_ds = train_raw.map(format_fn, remove_columns=remove_columns,num_proc=num_cpu_proc)
eval_ds = eval_raw.map(format_fn, remove_columns=getattr(eval_raw, "column_names", None),num_proc=num_cpu_proc)

#print (train_ds[0])

tokenize_fn = partial(tokenize_function, tokenizer=tokenizer)
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"], num_proc=num_cpu_proc)

#print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))

eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"], num_proc=num_cpu_proc)

if args.max_train_samples is not None and not args.streaming:
    train_ds = train_ds.select(range(min(args.max_train_samples, len(train_ds))))
if args.max_eval_samples is not None and not args.streaming:
    eval_ds = eval_ds.select(range(min(args.max_eval_samples, len(eval_ds))))

print ("group_fn, running in multiple processor so speed is fast but won't display the progress")
group_fn = partial(group_texts, block_size=args.block_size)
train_ds = train_ds.map(group_fn, num_proc=num_cpu_proc) #, batched=True will error
eval_ds = eval_ds.map(group_fn, num_proc=num_cpu_proc) #, batched=True will error

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
  

group_fn, running in multiple processor so speed is fast but won't display the progress


Map (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
config = RetNetConfig(
    vocab_size=len(tokenizer),
    hidden_size=args.hidden_size,
    intermediate_size=args.intermediate_size,
    num_hidden_layers=args.num_hidden_layers,
    num_retention_heads=args.num_retention_heads,
    value_dim=args.value_dim,
    hidden_dropout_prob=args.dropout,
    activation_dropout=args.activation_dropout,
    drop_path_rate=args.drop_path_rate,
    recurrent_chunk_size=args.recurrent_chunk_size,
    chunkwise_recurrent=args.chunkwise_recurrent,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))

Generate config GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 50256,
  "use_cache": true
}

You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embedding dimension will be 50257. This might induce some performance reduction as *Tensor Cores* will not be available. For more details about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc


Embedding(50257, 1024)

In [ ]:
#del trainer
#del model
#torch.cuda.empty_cache()

In [ ]:
training_args = TrainingArguments(
    output_dir=args.output_dir,
    #overwrite_output_dir=True,
    save_total_limit=5,
    do_train=True,
    do_eval=True,
    eval_strategy ="steps",
    eval_steps=args.eval_steps,
    save_steps=args.save_steps,
    logging_steps=args.logging_steps,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    weight_decay=args.weight_decay,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    warmup_ratio=args.warmup_ratio,
    lr_scheduler_type="cosine",
    bf16=args.bf16,
    fp16=args.fp16,
    push_to_hub=args.push_to_hub,
    hub_model_id=args.hub_model_id,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

PyTorch: setting up devices
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
transformers.logging.set_verbosity_info()

In [ ]:
train_result = trainer.train()
model = trainer.model
trainer.save_model(args.output_dir)
tokenizer.save_pretrained(args.output_dir)

metrics = trainer.evaluate()
if "eval_loss" in metrics:
    metrics["eval_perplexity"] = math.exp(metrics["eval_loss"])

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)
trainer.save_state()

print("Saved model to", os.path.abspath(args.output_dir))
print("Eval metrics:", metrics)

***** Running training *****
  Num examples = 50
  Num Epochs = 3
  Instantaneous batch size per device = 14
  Total train batch size (w. parallel, distributed & accumulation) = 28
  Gradient Accumulation steps = 2
  Total optimization steps = 6
  Number of trainable parameters = 143,755,264


ValueError: too many values to unpack (expected 3)

In [ ]:
prompt = """SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal
year does not become law prior to the beginning of these fiscal years
or a joint resolution making continuing appropriations is not in
effect, there is appropriated, out of any moneys in the Treasury not
otherwise appropriated, and out of applicable corporate or other
revenues, receipts, and funds, such sums as may be necessary to
continue any program, project, or activity for which funds were
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and
authority granted, for a program, project, or activity for any fiscal
year pursuant to this Act shall be at 100 percent of the rate of
operations that was provided for the program, project, or activity in
fiscal year 1999 in the corresponding regular appropriation Act for
fiscal year 1999.
    (c) Period of Availability.--Appropriations and funds made
available, and authority granted, for any fiscal year pursuant to this
Act for a program, project, or activity shall be available for the
period beginning with the first day of a lapse in appropriations and
ending with the earlier of--
            (1) the date on which the applicable regular appropriation
        bill for any fiscal year becomes law (whether or not that law
        provides for that program, project, or activity) or a
        continuing resolution making appropriations becomes law, as the
        case may be; or
            (2) the last day of any fiscal year.

SEC. 3. TERMS AND CONDITIONS.

    (a) In General.--An appropriation of funds made available, or
authority granted, for a program, project, or activity for any fiscal
year pursuant to this Act shall be made available to the extent and in
the manner which would be provided by the pertinent appropriations Act
for fiscal year 1999, including all of the terms and conditions and the
apportionment schedule imposed with respect to the appropriation made
or funds made available for fiscal year 1999 or authority granted for
the program, project, or activity under current law.
    (b) Extent and Manner.--Appropriations made by this Act shall be
available to the extent and in the manner which would be provided by
the pertinent appropriations Act.

SEC. 4. COVERAGE.

    Appropriations and funds made available, and authority granted, for
any program, project, or activity for any fiscal year pursuant to this
Act shall cover all obligations or expenditures incurred for that
program, project, or activity during the portion of any fiscal year for
which this Act applies to that program, project, or activity.

SEC. 5. EXPENDITURES.

    Expenditures made for a program, project, or activity for any
fiscal year pursuant to this Act shall be charged to the applicable
appropriation, fund, or authorization whenever a regular appropriation
bill or a joint resolution making continuing appropriations until the
end of any fiscal year providing for that program, project, or activity
for that period becomes law.

SEC. 6. INITIATING OR RESUMING A PROGRAM, PROJECT, OR ACTIVITY.

    No appropriation or funds made available or authority granted
pursuant to this Act shall be used to initiate or resume any program,
project, or activity for which appropriations, funds, or other
authority were not available during fiscal year 1999.

SEC. 7. PROTECTION OF OTHER OBLIGATIONS.

    Nothing in this Act shall be construed to effect Government
obligations mandated by other law, including obligations with respect
to Social Security, Medicare, Medicaid, and veterans benefits.

SEC. 8. DEFINITION.

    In this Act, the term ``regular appropriation bill'' means any
annual appropriation bill making appropriations, otherwise making funds
available, or granting authority, for any of the following categories
of programs, projects, and activities:
            (1) Agriculture, rural development, and related agencies
        programs.
            (2) The Departments of Commerce, Justice, and State, the
        judiciary, and related agencies.
            (3) The Department of Defense.
            (4) The government of the District of Columbia and other
        activities chargeable in whole or in part against the revenues
        of the District.
            (5) The Departments of Labor, Health and Human Services,
        and Education, and related agencies.
            (6) The Departments of Veterans Affairs and Housing and
        Urban Development, and sundry independent agencies, boards,
        commissions, corporations, and offices.
            (7) Energy and water development.
            (8) Foreign assistance and related programs.
            (9) The Department of the Interior and related agencies.
            (10) Military construction.
            (11) The Department of Transportation and related agencies.
            (12) The Treasury Department, the U.S. Postal Service, the
        Executive Office of the President, and certain independent
        agencies.
            (13) The legislative branch.
Summary:"""

In [ ]:
# Prompt (match your training format!)
import torch
model.to("cuda")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#inputs = torch.tensor(train_ds[0]['input_ids'])[None,:].to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        #inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
        use_cache = True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
model.to("cuda")
#inputs = torch.tensor(train_ds[0]['input_ids'])[None,:].to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
        use_cache = True #True for RNN, False for Parallel
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
outputs[0]

In [ ]:
# SANITY CHECK CELL
"""import importlib
import sys
import retnet_hf
importlib.reload(sys.modules['retnet_hf'])
importlib.reload(retnet_hf)
# optional but safer if things are really stuck
sys.modules["retnet_hf"] = retnet_hf

del model
del trainer

model = retnet_hf.RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)
train_result = trainer.train("outputs/retnet-billsum/checkpoint-8567")


model.to("cuda")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        #inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
        #use_cache = False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=False))"""